In [1]:
import torch as t
import torch.nn as nn
import torch.nn.functional as F

from typing import Annotated

DIM_STATE = 4
DIM_ACTIONS = 2

# pi(a|s)
policy_model = nn.Sequential(
  nn.Linear(DIM_STATE, 64),
  nn.ReLU(),
  nn.Linear(64, DIM_ACTIONS),
  nn.Softmax(dim=-1)
)

# V(s)
v_model = nn.Sequential(
  nn.Linear(DIM_STATE, 64),
  nn.ReLU(),
  nn.Linear(64, 1),
)

In [2]:
def select_action(obs: Annotated[t.Tensor, "N_ENVS DIM_STATE"]) -> Annotated[t.Tensor, "N_ENVS"]:
  action_probs = policy_model(obs)
  dist = t.distributions.Categorical(action_probs)
  return dist.sample()

In [3]:
import gymnasium as gym
import einops
from tqdm import tqdm

N_EPISODES = 5000
N_ROLLOUT = 10
DISCOUNT = 0.999

lr = 3e-3  # this is alpha = step_size
p_optim = t.optim.Adam(policy_model.parameters(), lr=lr)
v_optim = t.optim.Adam(v_model.parameters(), lr=lr)

# Initialise the environment
env = gym.make("CartPole-v1")

for _ in tqdm(range(N_EPISODES)):
  # Episode:
  obs, info = env.reset()
  obs = t.tensor(obs).unsqueeze(0)

  done = False
  terminated = False
  n = 0
  while not done:

    # Rollout
    actions = []
    rewards = []
    states = []

    for i in range(N_ROLLOUT):
      # this is where you would insert your policy
      action = select_action(obs).item()

      next_obs, reward, terminated, truncated, info = env.step(action)

      actions.append(action)
      rewards.append(float(reward))
      states.append(obs)

      obs = t.tensor(next_obs).unsqueeze(0)
      done = terminated or truncated

      n += 1

      if done:
        break

    # obs contains s_{t+k}
    discounted_rewards = []
    for i in reversed(range(len(rewards))):
      r = rewards[i]
      if discounted_rewards:
        r += DISCOUNT*discounted_rewards[-1]
      discounted_rewards.append(r)
    discounted_rewards = discounted_rewards[::-1]

    p_optim.zero_grad()
    v_optim.zero_grad()

    w_loss = t.zeros(())
    p_loss = t.zeros(())
    T = len(states)

    with t.no_grad():
      v_T = t.zeros(()) if terminated else v_model(obs)

    for i in range(T):
      v_i = v_model(states[i])

      with t.no_grad():
        delta = discounted_rewards[i] + DISCOUNT**(T-i)*v_T - v_i

      w_loss = w_loss - DISCOUNT**(n - T + i)*delta*v_i

      action_probs = policy_model(states[i])
      dist = t.distributions.Categorical(action_probs)
      log_prob = dist.log_prob(t.tensor(actions[i]))
      p_loss = p_loss - DISCOUNT**(n - T + i)*delta*log_prob

    w_loss.backward()
    p_loss.backward()
    v_optim.step()
    p_optim.step()


env.close()

100%|██████████| 5000/5000 [16:33<00:00,  5.03it/s]


In [5]:
env = gym.make("CartPole-v1", render_mode="human")

obs, info = env.reset()
obs = t.tensor(obs).unsqueeze(0)

done = False
while not done:

  # this is where you would insert your policy
  action = select_action(obs).item()

  next_obs, reward, terminated, truncated, info = env.step(action)
  obs = t.tensor(next_obs).unsqueeze(0)
  done = terminated or truncated

  if done:
    break


env.close()